In [ ]:
from rdflib import Graph, Literal, Namespace, RDF, URIRef
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
# import nltk

C:\Users\Makai\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Download NLTK punkt tokenizer
# nltk.download('punkt')

# Load LLaMA tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B")
model.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:15<00:00,  3.87s/it]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (n

In [5]:
def generate_and_save_rdf(prompt, top_k=3, n_steps=5, output_file="branching_structure.ttl"):
    # Initialize an RDF graph and namespace
    g = Graph()
    EX = Namespace("http://example.org/")
    g.bind("ex", EX)

    # Tokenize the initial prompt
    tokens = tokenizer(prompt, return_tensors='pt')
    input_ids = tokens.input_ids

    # Track each token generated in each step
    current_nodes = [input_ids]  # Start with the initial prompt as the root node

    for step in range(n_steps):
        print(f"\nStep {step+1}/{n_steps}", end="\t")
        next_nodes = []  # Reset next_nodes for each step to only hold new branches

        for branch_id, branch_input_ids in enumerate(current_nodes):
            # Generate top-k next tokens for each current token/branch
            with torch.no_grad():
                print(f"Generating top-{top_k} next tokens for branch {branch_id+1}/{len(current_nodes)}", end="\t")
                outputs = model(branch_input_ids)
                logits = outputs.logits[:, -1, :]

                # Ensure exactly top_k next tokens are selected
                top_k_probs, top_k_indices = torch.topk(logits, top_k, dim=-1)

                # Iterate over each of the top-k options and create branches
                for k in range(top_k):
                    print(f"Branch {branch_id+1}/{len(current_nodes)} - Option {k+1}/{top_k}", end="\t")
                    next_token = top_k_indices[0, k].item()
                    next_token_text = tokenizer.decode([next_token], skip_special_tokens=True)
                    new_input_ids = torch.cat([branch_input_ids, torch.tensor([[next_token]])], dim=-1)

                    # Create URIs for the current and next nodes
                    current_uri = EX[f"Branch_{step}_{branch_id}"]
                    next_uri = EX[f"Branch_{step+1}_{branch_id}_{k}"]

                    # Add current token's branch node if not already present
                    if (current_uri, RDF.type, EX.Path) not in g:
                        g.add((current_uri, RDF.type, EX.Path))
                        g.add((current_uri, EX.text, Literal(tokenizer.decode(branch_input_ids[0], skip_special_tokens=True))))
                        g.add((current_uri, EX.iteration, Literal(step)))

                    # Add next token node
                    g.add((next_uri, RDF.type, EX.Token))
                    g.add((next_uri, EX.text, Literal(next_token_text)))
                    g.add((next_uri, EX.iteration, Literal(step + 1)))
                    # Link the current token to the next token
                    g.add((current_uri, EX.next, next_uri))

                    # Save this new token sequence for the next step
                    next_nodes.append(new_input_ids)

        # Update current_nodes for the next iteration to only include exactly top_k branches
        current_nodes = next_nodes

    # Save the entire RDF graph to a single Turtle file
    g.serialize(destination=output_file, format="turtle")
    print(f"\n\nSaved branching structure to RDF file: {output_file}")

In [9]:
# Example usage
prompt = "Once upon a time in a distant land"
output = generate_and_save_rdf(prompt, top_k=4, n_steps=4)


Step 1/4	Generating top-4 next tokens for branch 1/1	Branch 1/1 - Option 1/4	Branch 1/1 - Option 2/4	Branch 1/1 - Option 3/4	Branch 1/1 - Option 4/4	
Step 2/4	Generating top-4 next tokens for branch 1/4	Branch 1/4 - Option 1/4	Branch 1/4 - Option 2/4	Branch 1/4 - Option 3/4	Branch 1/4 - Option 4/4	Generating top-4 next tokens for branch 2/4	Branch 2/4 - Option 1/4	Branch 2/4 - Option 2/4	Branch 2/4 - Option 3/4	Branch 2/4 - Option 4/4	Generating top-4 next tokens for branch 3/4	Branch 3/4 - Option 1/4	Branch 3/4 - Option 2/4	Branch 3/4 - Option 3/4	Branch 3/4 - Option 4/4	Generating top-4 next tokens for branch 4/4	Branch 4/4 - Option 1/4	Branch 4/4 - Option 2/4	Branch 4/4 - Option 3/4	Branch 4/4 - Option 4/4	
Step 3/4	Generating top-4 next tokens for branch 1/16	Branch 1/16 - Option 1/4	Branch 1/16 - Option 2/4	Branch 1/16 - Option 3/4	Branch 1/16 - Option 4/4	Generating top-4 next tokens for branch 2/16	Branch 2/16 - Option 1/4	Branch 2/16 - Option 2/4	Branch 2/16 - Option 3/4	Branc